In [3]:
# Importar las librerías necesarias
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Definir la ruta de la carpeta de datos y crear rutas para guardar resultados y gráficos
folder_path = r"D:\ORDENADOR AFM1\BACTERIAS EXTREMÓFILAS\64.Mediciones Helios desecadas 2024-08-22\64.5.Mediciones Helios-Exp-09-09-2024\AFM Data1"
results_path = os.path.join(folder_path, "Resultados Análisis")
os.makedirs(results_path, exist_ok=True)  # Crear carpeta para resultados si no existe
plots_path = os.path.join(results_path, "Plots")
os.makedirs(plots_path, exist_ok=True)  # Crear carpeta para gráficos si no existe

# Inicializar un DataFrame vacío para almacenar todos los resultados
results_df = pd.DataFrame(columns=[
    "Archivo", "k cantilever (N/m)", "Pendiente SiO2 (nm/V)",
    "Pendiente bacteria (nm/V)", "Pendiente bacteria (V/nm)",
    "Rigidez bacteria (N/m)", "Indentación (nm)"
])

# Constantes del análisis
kcantilver = 38.0  # Constante de rigidez del cantiléver
pendiente_superficie_infinita_dura = 45.0  # Pendiente para superficie infinitamente dura (referencia)

# Función para determinar el tamaño de ventana inicial óptimo
def find_optimal_window_size(F_data, delta_data, min_size=30, max_size=140, step=5):
    slopes = []  # Lista para almacenar las pendientes calculadas para cada ventana
    for size in range(min_size, max_size + 1, step):
        x_window = delta_data[:size]
        y_window = F_data[:size]
        try:
            # Ajuste lineal de los datos en la ventana actual
            params, _ = curve_fit(lambda x, a, b: a * x + b, x_window, y_window)
            slope = abs(params[0])  # Calcular la pendiente absoluta
            slopes.append((size, slope))  # Guardar el tamaño de la ventana y la pendiente
        except RuntimeError:
            continue
    # Seleccionar el tamaño de ventana con la pendiente más baja
    optimal_window = min(slopes, key=lambda x: x[1])[0]
    return optimal_window

# Iterar sobre cada archivo en la carpeta de datos que coincida con el formato esperado
for filename in os.listdir(folder_path):
    if filename.endswith('Normal force.fz.cur'):
        file_path = os.path.join(folder_path, filename)
        print(f"Analizando archivo: {filename}")

        # Leer los datos del archivo y extraer la sección de datos
        with open(file_path, 'r') as f:
            lines = f.readlines()

        # Identificar dónde comienzan los datos numéricos, después del encabezado
        data_start_index = lines.index('[Header end]\n') + 1
        data_lines = lines[data_start_index:]
        # Convertir las líneas de datos en una matriz NumPy
        data = [list(map(float, line.strip().split())) for line in data_lines if line.strip() and not line.startswith('[')]

        # Separar los datos en desplazamiento (delta_data) y fuerza (F_data)
        data_array = np.array(data)
        delta_data = data_array[:, 0]  # Desplazamiento en nanómetros (nm)
        F_data = data_array[:, 1]      # Fuerza en Voltios (V)

        # Encontrar el tamaño de ventana inicial óptimo para la región de no contacto
        initial_window_size = find_optimal_window_size(F_data, delta_data)
        # Calcular la media y desviación estándar en la región inicial
        mean_force = np.mean(F_data[:initial_window_size])
        std_force = np.std(F_data[:initial_window_size])
        threshold = std_force * 2  # Umbral para detectar el fin de la región de no contacto

        # Identificar el fin de la región de no contacto
        for end_index in range(initial_window_size, len(F_data)):
            if np.abs(F_data[end_index] - mean_force) > threshold:
                no_contact_region_end = end_index
                break
        else:
            no_contact_region_end = len(F_data)  # En caso de no encontrar un cambio significativo

        # Realizar un ajuste lineal para la región de no contacto
        region_no_contact = delta_data[:no_contact_region_end]
        F_no_contact = F_data[:no_contact_region_end]
        params, _ = curve_fit(lambda x, a, b: a * x + b, region_no_contact, F_no_contact)
        a_fit, b_fit = params  # Coeficientes del ajuste lineal

        # Normalización de la curva de fuerza restando el ajuste lineal
        F_base = a_fit * delta_data + b_fit
        F_normalized = F_data - F_base
        F_normalized -= F_normalized[0]  # Alinear la curva normalizada a cero

        # Identificar el último cruce por cero en la curva normalizada
        zero_crossing_indices = np.where(np.diff(np.sign(F_normalized)))[0]

        # Si hay cruces por cero, proceder con el análisis de la región de contacto
        if zero_crossing_indices.size > 0:
            last_zero_crossing_index = zero_crossing_indices[-1] + 1
            delta_contact = delta_data[last_zero_crossing_index:]
            F_contact = F_normalized[last_zero_crossing_index:]

            best_r2 = -1  # Valor inicial de R² (coeficiente de determinación)
            best_params = None
            best_segment = None
            min_window_size = 15  # Tamaño mínimo de ventana para el ajuste lineal
            max_window_size = len(F_contact)

            # Buscar el mejor ajuste lineal en la región de contacto
            for window_size in range(min_window_size, max_window_size + 1):
                for start in range(len(F_contact) - window_size + 1):
                    end = start + window_size
                    delta_segment = delta_contact[start:end]
                    F_segment = F_contact[start:end]
                    try:
                        params_segment, _ = curve_fit(lambda x, a, b: a * x + b, delta_segment, F_segment)
                        a_fit_segment, b_fit_segment = params_segment
                        F_pred = a_fit_segment * delta_segment + b_fit_segment
                        r2 = 1 - (np.sum((F_segment - F_pred) ** 2) / np.sum((F_segment - np.mean(F_segment)) ** 2))
                        if r2 > best_r2:
                            best_r2 = r2
                            best_params = params_segment
                            best_segment = (start, end)
                    except Exception:
                        continue

            # Si se encontró un ajuste adecuado
            if best_params is not None:
                a_fit_best, b_fit_best = best_params
                F_contact_fit_best = a_fit_best * delta_contact[best_segment[0]:best_segment[1]] + b_fit_best

                # Calcular las constantes de rigidez y otros parámetros importantes
                stiffness_V_per_nm = a_fit_best  # Rigidez en V/nm
                stiffness_nm_per_V = 1 / stiffness_V_per_nm if stiffness_V_per_nm != 0 else np.inf  # Rigidez inversa
                k_bacteria = kcantilver * ((stiffness_V_per_nm * pendiente_superficie_infinita_dura) / (1 - (stiffness_V_per_nm * pendiente_superficie_infinita_dura)))

                # Análisis de la indentación ajustada
                indentation_start_index = last_zero_crossing_index
                indentation_data = delta_data[indentation_start_index:]
                F_indentation = F_normalized[indentation_start_index:]
                F_indentation_nN = F_indentation * kcantilver * pendiente_superficie_infinita_dura  # Fuerza en nN
                indentation_adjusted = indentation_data - (pendiente_superficie_infinita_dura * F_indentation)  # Indentación ajustada
                total_indentacion_nm = indentation_adjusted[-1] - indentation_adjusted[0]  # Indentación total ajustada

                # Guardar los resultados en el DataFrame
                result_row = pd.DataFrame([{
                    "Archivo": filename,
                    "k cantilever (N/m)": kcantilver,
                    "Pendiente SiO2 (nm/V)": pendiente_superficie_infinita_dura,
                    "Pendiente bacteria (nm/V)": stiffness_nm_per_V,
                    "Pendiente bacteria (V/nm)": stiffness_V_per_nm,
                    "Rigidez bacteria (N/m)": k_bacteria,
                    "Indentación (nm)": total_indentacion_nm
                }])

                results_df = pd.concat([results_df, result_row], ignore_index=True)

                # Graficar los resultados y guardar en la carpeta de Plots
                plt.figure(figsize=(15, 5))

                # Gráfico de la curva original
                plt.subplot(1, 3, 1)
                plt.plot(delta_data, F_data, label='Curva Original', color='blue')
                plt.axhline(0, color='gray', linestyle='--')
                plt.xlabel('Desplazamiento (nm)')
                plt.ylabel('Fuerza (V)')
                plt.title('Curva Original')
                plt.legend()

                # Gráfico de la curva normalizada y ajuste
                plt.subplot(1, 3, 2)
                plt.plot(delta_data, F_normalized, label='Curva Normalizada', color='orange')
                plt.plot(delta_contact[best_segment[0]:best_segment[1]], F_contact_fit_best, label='Mejor Ajuste', color='red')
                plt.axhline(0, color='gray', linestyle='--')
                plt.xlabel('Desplazamiento (nm)')
                plt.ylabel('Fuerza Normalizada (V)')
                plt.title('Curva Normalizada y Ajuste de Contacto')
                # Añadir anotación dentro del gráfico
                plt.text(0.2, 0.8, f"Pendiente: {stiffness_nm_per_V:.2f} nm/V", transform=plt.gca().transAxes,
                         verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
                plt.legend()

                # Gráfico del análisis de indentación
                plt.subplot(1, 3, 3)
                plt.plot(indentation_adjusted, F_indentation_nN, label='Indentación Ajustada', color='green')
                plt.axhline(0, color='gray', linestyle='--')
                plt.xlabel('Indentación Ajustada (nm)')
                plt.ylabel('Fuerza de Indentación (nN)')
                plt.title('Análisis de Indentación')
                # Añadir anotación dentro del gráfico
                plt.text(0.2, 0.8, f"Indentación: {total_indentacion_nm:.3f} nm", transform=plt.gca().transAxes,
                         verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
                plt.legend()

                # Ajustar el diseño y guardar los gráficos
                plt.tight_layout()
                plot_filename = os.path.join(plots_path, f"{os.path.splitext(filename)[0]}_plots.png")
                plt.savefig(plot_filename)
                plt.close()
                print(f"Gráfico guardado en: {plot_filename}")
            else:
                print(f"No se encontró un ajuste adecuado en la región de contacto para el archivo: {filename}")
        else:
            print(f"No se encontró un paso por cero en la curva normalizada para el archivo: {filename}")

# Guardar todos los resultados en un archivo Excel
excel_filename = os.path.join(results_path, "Resultados_Analisis.xlsx")
results_df.to_excel(excel_filename, index=False)
print("Análisis completado. Resultados y gráficos guardados en la carpeta 'Resultados Análisis'.")


Analizando archivo: 64.5-Helios-desecacion-exponencial-09-09-2024_0001_Normal force.fz.cur


C:\Users\Monica Luna\AppData\Local\Temp\ipykernel_7672\2960520313.py:152: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, result_row], ignore_index=True)


Gráfico guardado en: D:\ORDENADOR AFM1\BACTERIAS EXTREMÓFILAS\64.Mediciones Helios desecadas 2024-08-22\64.5.Mediciones Helios-Exp-09-09-2024\AFM Data1\Resultados Análisis\Plots\64.5-Helios-desecacion-exponencial-09-09-2024_0001_Normal force.fz_plots.png
Analizando archivo: 64.5-Helios-desecacion-exponencial-09-09-2024_0002_Normal force.fz.cur
Gráfico guardado en: D:\ORDENADOR AFM1\BACTERIAS EXTREMÓFILAS\64.Mediciones Helios desecadas 2024-08-22\64.5.Mediciones Helios-Exp-09-09-2024\AFM Data1\Resultados Análisis\Plots\64.5-Helios-desecacion-exponencial-09-09-2024_0002_Normal force.fz_plots.png
Analizando archivo: 64.5-Helios-desecacion-exponencial-09-09-2024_0003_Normal force.fz.cur
Gráfico guardado en: D:\ORDENADOR AFM1\BACTERIAS EXTREMÓFILAS\64.Mediciones Helios desecadas 2024-08-22\64.5.Mediciones Helios-Exp-09-09-2024\AFM Data1\Resultados Análisis\Plots\64.5-Helios-desecacion-exponencial-09-09-2024_0003_Normal force.fz_plots.png
Analizando archivo: 64.5-Helios-desecacion-exponencia